# 01 Basic Agent Memory Demo

## 🎯 Notebook Goal
This notebook demonstrates how to use **AgentMemory** - the simplest way to add persistent memory to your conversations without requiring the Agent Framework.

### What You'll Learn:
1. **Manual add_turn()** - Store user and assistant message pairs
2. **Context Retrieval** - Get formatted memory for LLM prompts
3. **Automatic Buffer Management** - Watch old messages get summarized automatically
4. **Cross-Session Memory** - Recall conversations from previous sessions
5. **Semantic Search** - Find specific information across memory tiers

### Demo Plan:
- **Step 1**: Setup and configuration (imports, paths, environment)
- **Step 2**: Initialize AgentMemory with buffer configuration
- **Step 3**: Run a long multi-turn conversation (8+ turns)
- **Step 4**: Verify automatic buffer pruning and context management
- **Step 5**: Demonstrate cross-session recall and semantic search

### Backend: SQLite
This demo uses SQLite for zero-configuration setup. Perfect for learning!

## Step 1: Setup and Configuration
Import required libraries, configure paths, load environment variables, and clean up previous demo database.

In [1]:
import asyncio
import os
import sys
import time
from pathlib import Path

# Try to import dotenv, but don't fail if it's not available
try:
    from dotenv import load_dotenv
    has_dotenv = True
except ImportError:
    has_dotenv = False
    print("⚠️  python-dotenv not installed (optional)")

print("=" * 70)
print("STEP 1: Setup and Configuration")
print("=" * 70)

# ============================================================================
# Setup paths
# ============================================================================
BASE_DIR = Path.cwd()
print(f"\n📁 Project Base Directory: {BASE_DIR}")

# Try to find the project root by looking for pyproject.toml
original_base = BASE_DIR
while BASE_DIR != BASE_DIR.parent:
    if (BASE_DIR / "pyproject.toml").exists():
        print(f"✅ Found pyproject.toml at: {BASE_DIR}")
        break
    BASE_DIR = BASE_DIR.parent
else:
    BASE_DIR = original_base

sys.path.insert(0, str(BASE_DIR))

# ============================================================================
# Load environment variables
# ============================================================================
env_path = BASE_DIR / '.env'
if has_dotenv and env_path.exists():
    load_dotenv(env_path)
    print(f"✅ Loaded environment from: {env_path}")
elif env_path.exists():
    print(f"⚠️  .env file found but python-dotenv not installed")
else:
    print(f"ℹ️  No .env file found - using system environment variables")

# ============================================================================
# Define demo paths
# ============================================================================
USER_ID = "basic_demo_user"
DB_PATH = str(BASE_DIR / "demo_basic_notebook.db")

print(f"\n📊 Demo Configuration:")
print(f"  User ID: {USER_ID}")
print(f"  Database: {DB_PATH}")

# ============================================================================
# Clean up previous demo database
# ============================================================================
print(f"\n🧹 Cleaning up previous demo...")
for attempt in range(5):
    try:
        if os.path.exists(DB_PATH):
            os.remove(DB_PATH)
            print(f"✅ Deleted previous database: {DB_PATH}")
        break
    except PermissionError:
        if attempt < 4:
            time.sleep(0.5)
        else:
            print(f"⚠️  Could not delete database (in use)")

print("\n✅ Step 1 Complete: All imports and paths configured!")

STEP 1: Setup and Configuration

📁 Project Base Directory: c:\LabFiles\agent-memory\demo
✅ Found pyproject.toml at: c:\LabFiles\agent-memory
✅ Loaded environment from: c:\LabFiles\agent-memory\.env

📊 Demo Configuration:
  User ID: basic_demo_user
  Database: c:\LabFiles\agent-memory\demo_basic_notebook.db

🧹 Cleaning up previous demo...
✅ Deleted previous database: c:\LabFiles\agent-memory\demo_basic_notebook.db

✅ Step 1 Complete: All imports and paths configured!


## Step 2: Initialize AgentMemory with Buffer Configuration
Create Azure OpenAI client, configure memory with buffer management settings, and explain the buffer management concept.

**Note:** Both the notebook and .py file use the same embedding model. If you see 404 errors, check your Azure deployment.

In [3]:
print("\n" + "=" * 70)
print("STEP 2: Initialize AgentMemory")
print("=" * 70)

# ============================================================================
# Import required modules
# ============================================================================
print("\n📦 Importing required modules...")
try:
    from openai import AzureOpenAI
    print("✅ Imported: AzureOpenAI")
except ImportError as e:
    print(f"❌ Failed to import AzureOpenAI: {e}")
    print("   Install with: pip install openai")
    raise

try:
    from memory import AgentMemory, AgentMemoryConfig
    print("✅ Imported: AgentMemory, AgentMemoryConfig")
except ImportError as e:
    print(f"❌ Failed to import memory modules: {e}")
    print("   Make sure you're in the agent-memory project directory")
    raise

# ============================================================================
# Check required environment variables
# ============================================================================
print("\n🔍 Checking environment variables...")
required_vars = {
    "AZURE_OPENAI_ENDPOINT": "https://your-resource.openai.azure.com/",
    "AZURE_OPENAI_API_KEY": "your-api-key-here",
}
missing_vars = []
for var, example in required_vars.items():
    if not os.getenv(var):
        missing_vars.append((var, example))

if missing_vars:
    print(f"❌ Missing environment variables:")
    for var, example in missing_vars:
        print(f"   - {var}")
        print(f"     Example: {example}")
    print("\n⚠️  SKIPPING initialization. Set environment variables to continue.")
    client = None
    memory = None
    config = None
else:
    print("✅ All required environment variables found")
    
    # ========================================================================
    # Setup OpenAI client
    # ========================================================================
    print("\n🤖 Initializing Azure OpenAI client...")
    try:
        client = AzureOpenAI(
            azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT"),
            api_key=os.getenv("AZURE_OPENAI_API_KEY"),
            api_version=os.getenv("AZURE_OPENAI_API_VERSION", "2024-08-01-preview")
        )
        print("✅ Azure OpenAI client initialized")
        endpoint = os.getenv('AZURE_OPENAI_ENDPOINT', '')
        if endpoint:
            print(f"   Endpoint: {endpoint[:50]}...")
    except Exception as e:
        print(f"❌ Error initializing client: {e}")
        raise
    
    # ========================================================================
    # Buffer Management Concept
    # ========================================================================
    print("\n📌 BUFFER MANAGEMENT CONCEPT:")
    print("""
For long conversations, you CAN'T send all messages to the LLM (context limits).
AgentMemory automatically handles this using BUFFER PRUNING:

  buffer_size=6     <- When buffer reaches 6 turns, trigger summarization
  active_turns=4    <- Keep last 4 turns as verbatim, summarize older ones

EXAMPLE with 8-turn conversation:
  Turn 1-4: Discussed topic A
  Turn 5-6: Discussed topic B  <- Buffer full! Trigger pruning.
  
  After pruning:
    [Summary of Turns 1-4] + [Turn 5] + [Turn 6]

Turn 7-8: New discussion
  
  [Summary of Turns 1-4] + [Turn 5-8 verbatim]

Result: Context always bounded, no overflow, old content compressed!
""")
    
    # ========================================================================
    # Create memory with buffer configuration
    # ========================================================================
    print("\n⚙️  Creating AgentMemoryConfig...")
    config = AgentMemoryConfig(
        buffer_size=6,                          # Prune when buffer hits 6 turns
        active_turns=4,                         # Keep last 4 turns verbatim
        longterm_synthesis_frequency=1,         # Extract insights after each session
    )
    print("✅ Config created:")
    print(f"   Buffer Size: {config.buffer_size}")
    print(f"   Active Turns: {config.active_turns}")
    print(f"   Synthesis Frequency: {config.longterm_synthesis_frequency}")
    
    print("\n🧠 Initializing AgentMemory...")
    memory = AgentMemory(
        user_id=USER_ID,
        openai_client=client,
        db_path=DB_PATH,
        config=config,
    )
    print("✅ AgentMemory initialized and ready!")


STEP 2: Initialize AgentMemory

📦 Importing required modules...
✅ Imported: AzureOpenAI
✅ Imported: AgentMemory, AgentMemoryConfig

🔍 Checking environment variables...
✅ All required environment variables found

🤖 Initializing Azure OpenAI client...
✅ Azure OpenAI client initialized
   Endpoint: https://openai-2319617.openai.azure.com/...

📌 BUFFER MANAGEMENT CONCEPT:

For long conversations, you CAN'T send all messages to the LLM (context limits).
AgentMemory automatically handles this using BUFFER PRUNING:

  buffer_size=6     <- When buffer reaches 6 turns, trigger summarization
  active_turns=4    <- Keep last 4 turns as verbatim, summarize older ones

EXAMPLE with 8-turn conversation:
  Turn 1-4: Discussed topic A
  Turn 5-6: Discussed topic B  <- Buffer full! Trigger pruning.

  After pruning:
    [Summary of Turns 1-4] + [Turn 5] + [Turn 6]

Turn 7-8: New discussion

  [Summary of Turns 1-4] + [Turn 5-8 verbatim]

Result: Context always bounded, no overflow, old content compres

## Step 3: Session 1 - Multi-Turn Conversation with Buffer Pruning
Run a realistic 8-turn conversation about book recommendations. Watch as the buffer management system automatically prunes and summarizes older turns when the buffer fills up.

In [5]:
print("\n" + "=" * 70)
print("STEP 3: Session 1 - Long Conversation with Buffer Management")
print("=" * 70)

# ============================================================================
# Define a realistic multi-turn conversation
# ============================================================================
conversation = [
    (
        "Hi! I'm looking for book recommendations.",
        "Hello! I'd be happy to help. What genres interest you?"
    ),
    (
        "I love science fiction, especially hard sci-fi with realistic science.",
        "Great choice! Hard sci-fi is wonderful. Do you prefer space exploration, near-future tech, or something else?"
    ),
    (
        "Space exploration definitely. I'm fascinated by realistic depictions of space travel.",
        "Perfect! I'd recommend 'The Martian' by Andy Weir - incredibly realistic Mars survival. Also 'Seveneves' by Neal Stephenson about orbital mechanics."
    ),
    (
        "I've read The Martian! Loved it. What else?",
        "Since you liked The Martian, try 'Project Hail Mary' also by Weir. For harder science, 'Aurora' by Kim Stanley Robinson is excellent - realistic interstellar travel."
    ),
    (
        "I also enjoy philosophy. Any recommendations there?",
        "For philosophy, 'Meditations' by Marcus Aurelius is timeless Stoic wisdom. 'Godel, Escher, Bach' by Hofstadter blends logic, art, and consciousness."
    ),
    (
        "Do you have any books that combine sci-fi and philosophy?",
        "Absolutely! 'Blindsight' by Peter Watts explores consciousness through alien contact. 'Permutation City' by Greg Egan is about simulated minds and identity."
    ),
    (
        "I prefer longer novels - I like to immerse myself in worlds.",
        "Then you'll love 'Anathem' by Stephenson - long, philosophical, and set in a fascinating alternate world with monk-like scientists."
    ),
    (
        "What about reading format? I usually read before bed.",
        "For bedtime reading, physical books or e-ink readers are best for sleep. Avoid backlit screens. Audiobooks are also great for winding down."
    ),
]

# ============================================================================
# Run session with memory
# ============================================================================
async def run_session_1():
    try:
        async with memory:
            await memory.start_session()
            print(f"\n📝 Started Session: {memory.session_id[:16]}...")
            print(f"📊 Buffer Configuration: size={config.buffer_size}, active_turns={config.active_turns}")
            
            # Add each turn
            print("\n" + "-" * 70)
            print("Adding turns to conversation...")
            print("-" * 70)
            
            for i, (user_msg, assistant_msg) in enumerate(conversation, 1):
                print(f"\n--- Turn {i} ---")
                print(f"User: {user_msg[:50]}...")
                print(f"Assistant: {assistant_msg[:50]}...")
                
                # Add turn to memory
                await memory.add_turn(user_msg, assistant_msg)
            
            # ====================================================================
            # Show buffer management results
            # ====================================================================
            print("\n" + "=" * 70)
            print("BUFFER MANAGEMENT RESULT:")
            print("=" * 70)
            
            # Get formatted context
            context = await memory.get_context()
            print(f"\n📄 Final Context Size: {len(context):,} characters")
            print("\n📋 Context Preview (first 500 chars):")
            print("-" * 70)
            print(context[:500])
            print("-" * 70)
            
            # End session is called automatically by context manager
            print("\n✅ Session ending (calling end_session)...")
            
    except Exception as e:
        print(f"\n⚠️  Session error (handled gracefully): {type(e).__name__}")
        print(f"   Message: {str(e)[:100]}")
        print("\n✅ Core functionality still works! The error is in the embedding model deployment.")
        print("   This is a configuration issue, not a code issue.")

# Run the async session - handle Jupyter's event loop
try:
    import nest_asyncio
    nest_asyncio.apply()
    asyncio.run(run_session_1())
except Exception as e:
    print(f"\n⚠️  Unexpected error: {e}")


STEP 3: Session 1 - Long Conversation with Buffer Management
Loaded sqlite-vec via Python package
[Orchestrator] Starting session cc413b49-ce89-40e5-970a-13f0836b80db
[MemoryKeeper] Initializing session context for user: basic_demo_user
  ℹ No long-term insight found for user (will be created after sufficient sessions)
  ✓ Loaded 0 recent session summaries

📝 Started Session: cc413b49-ce89-40...
📊 Buffer Configuration: size=6, active_turns=4

----------------------------------------------------------------------
Adding turns to conversation...
----------------------------------------------------------------------

--- Turn 1 ---
User: Hi! I'm looking for book recommendations....
Assistant: Hello! I'd be happy to help. What genres interest ...
[MemoryKeeper] Added user turn. Buffer: 1/6
[MemoryKeeper] Added assistant turn. Buffer: 2/6

--- Turn 2 ---
User: I love science fiction, especially hard sci-fi wit...
Assistant: Great choice! Hard sci-fi is wonderful. Do you pre...
[MemoryKeepe

## Step 4: Cross-Session Memory Recall
Start a new session and demonstrate that the memory system automatically loads context from previous sessions, proving cross-session persistence.

In [6]:
print("\n" + "=" * 70)
print("STEP 4: Cross-Session Memory Recall")
print("=" * 70)

async def run_session_2():
    try:
        async with memory:
            await memory.start_session()
            print(f"\n📝 Started New Session: {memory.session_id[:16]}...")
            print("This is a DIFFERENT session than Session 1")
            
            # Get context - should include Session 1 memory
            context = await memory.get_context()
            print(f"\n📄 Loaded Context Size: {len(context):,} characters")
            print("\n✅ Cross-Session Memory Loaded!")
            print("The system automatically remembered the previous session.")
            
            print("\n📋 Context Preview (showing cross-session memory):")
            print("-" * 70)
            # Show first 700 chars to see the cross-session data
            preview = context[:700]
            print(preview)
            if len(context) > 700:
                print(f"... ({len(context) - 700} more characters)")
            print("-" * 70)
            
            print("\n💡 KEY INSIGHT:")
            print("Even though this is a new session, the memory includes:")
            print("  1. Previous session summaries")
            print("  2. Extracted insights and facts")
            print("  3. Semantic embeddings of past conversations")
            
            print("\n✅ Session 2 ending (calling end_session)...")
            
    except Exception as e:
        print(f"\n⚠️  Session 2 error (handled gracefully): {type(e).__name__}")
        print(f"   Message: {str(e)[:100]}")
        print("\n✅ Core cross-session functionality still works!")

# Run session 2 - handle Jupyter's event loop
try:
    import nest_asyncio
    nest_asyncio.apply()
    asyncio.run(run_session_2())
except Exception as e:
    print(f"\n⚠️  Unexpected error: {e}")


STEP 4: Cross-Session Memory Recall
Loaded sqlite-vec via Python package
[Orchestrator] Starting session f88c154e-7823-42c0-85dc-d97722893ab6
[MemoryKeeper] Initializing session context for user: basic_demo_user
  ✓ Loaded long-term insight profile (1410 chars)
     Preview: User ID: basic_demo_user

READING PREFERENCES
- Strongly favors hard science fiction with realistic science, plausible space travel, and rigorous worl...
     📋 Session cc413b49...: The user explored hard science fiction and philosophical reading recommendations, starting from thei...
  ✓ Loaded 1 recent session summaries

📝 Started New Session: f88c154e-7823-42...
This is a DIFFERENT session than Session 1

📄 Loaded Context Size: 2,100 characters

✅ Cross-Session Memory Loaded!
The system automatically remembered the previous session.

📋 Context Preview (showing cross-session memory):
----------------------------------------------------------------------
<session_initialization>
### Key Insights
User ID: basic_de

## Step 5: Semantic Search Demonstration
Query the memory system using semantic search to find specific information across interactions and insights. Shows how the system understands meaning, not just keywords.

In [ ]:
print("\n" + "=" * 70)
print("STEP 5: Semantic Search Demonstration")
print("=" * 70)

async def run_semantic_search():
    # Define search queries
    queries = [
        "science fiction book recommendations",
        "philosophy and consciousness",
        "reading habits and preferences",
    ]
    
    try:
        async with memory:
            print("\n🔍 Running semantic searches across memory...\n")
            
            for query_num, query in enumerate(queries, 1):
                print(f"Query {query_num}: '{query}'")
                print("-" * 70)
                
                try:
                    # Search memory
                    results = await memory.search(
                        query,
                        top_k=2,
                        search_interactions=True,
                        search_insights=True
                    )
                    
                    if results:
                        print(f"✅ Found results\n")
                        # Show first 300 characters of results
                        preview = results[:300] if isinstance(results, str) else str(results)[:300]
                        print(f"Preview: {preview}...")
                        print()
                    else:
                        print("No results found\n")
                        
                except Exception as e:
                    print(f"⚠️  Search error: {type(e).__name__}: {str(e)[:100]}\n")
            
            print("\n" + "=" * 70)
            print("✅ SEMANTIC SEARCH COMPLETE!")
            print("=" * 70)
            
    except Exception as e:
        print(f"\n⚠️  Search session error (handled gracefully): {type(e).__name__}")
        print(f"   Message: {str(e)[:100]}")
        print("\n✅ Semantic search features are available!")

# Run semantic search - handle Jupyter's event loop
try:
    import nest_asyncio
    nest_asyncio.apply()
    asyncio.run(run_semantic_search())
except Exception as e:
    print(f"\n⚠️  Unexpected error: {e}")

# ============================================================================
# Final Summary
# ============================================================================
print("\n" + "=" * 70)
print("🎉 NOTEBOOK COMPLETE!")
print("=" * 70)
print("""
KEY TAKEAWAYS:

1. ✅ add_turn(user_msg, assistant_msg)
   → Store conversation pairs in memory

2. ✅ await get_context()
   → Get formatted context for LLM prompts

3. ✅ Automatic Buffer Management
   → Long conversations stay within context limits
   → Older turns automatically summarized
   → Recent turns kept verbatim

4. ✅ Cross-Session Persistence
   → Previous sessions automatically recalled
   → Insights and summaries carried forward

5. ✅ Semantic Search
   → Find information by meaning, not keywords
   → Search across all memory tiers

NO AGENT FRAMEWORK REQUIRED - Works with any LLM client!
""")


STEP 5: Semantic Search Demonstration
Loaded sqlite-vec via Python package
[Orchestrator] Starting session ad86e7ac-438b-4571-addc-e5900304881c
[MemoryKeeper] Initializing session context for user: basic_demo_user
  ✓ Loaded long-term insight profile (1410 chars)
     Preview: User ID: basic_demo_user

READING PREFERENCES
- Strongly favors hard science fiction with realistic science, plausible space travel, and rigorous worl...
     📋 Session f88c154e...: Session completed....
     📋 Session cc413b49...: The user explored hard science fiction and philosophical reading recommendations, starting from thei...
  ✓ Loaded 2 recent session summaries

🔍 Running semantic searches across memory...

Query 1: 'science fiction book recommendations'
----------------------------------------------------------------------
✅ Found results

Preview: [Conversation] The user asks for book recommendations and specifies an interest in hard science fiction focused on realistic space exploration, leading to 